# 04 — Frustum Extraction
For each YOLO detection, extract the LiDAR points that fall inside the 2D bounding box and estimate real-world distance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics opencv-python matplotlib -q

In [ ]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
from ultralytics import YOLO

BASE_PATH  = '/content/drive/MyDrive/Project/Kitti/Training'
IMAGE_PATH = os.path.join(BASE_PATH, 'image_2/training/image_2')
LIDAR_PATH = os.path.join(BASE_PATH, 'velodyne/training/velodyne')
CALIB_PATH = os.path.join(BASE_PATH, 'calib/data_object_calib/training/calib')

def parse_calib(calib_path):
    calib = {}
    with open(calib_path, 'r') as f:
        for line in f.readlines():
            if ':' in line:
                key, value = line.split(':', 1)
                calib[key.strip()] = np.array(
                    [float(x) for x in value.strip().split()])
    return (calib['P2'].reshape(3,4),
            calib['R0_rect'].reshape(3,3),
            calib['Tr_velo_to_cam'].reshape(3,4))

def project_lidar_to_image(points_3d, P2, R0, Tr):
    N       = points_3d.shape[0]
    pts_hom = np.hstack([points_3d, np.ones((N,1))])
    Tr_full = np.vstack([Tr,[0,0,0,1]])
    R0_full = np.eye(4); R0_full[:3,:3] = R0
    pts_cam = R0_full @ Tr_full @ pts_hom.T
    depth   = pts_cam[2,:]; valid = depth > 0
    pts_cam = pts_cam[:,valid]; depth = depth[valid]
    pts_img = P2 @ pts_cam; pts_img = pts_img/pts_img[2,:]
    return pts_img[:2,:].T, depth, valid

def get_lidar_points_in_box(box_2d, pixels, depth, pts_xyz, valid_mask):
    x1,y1,x2,y2 = box_2d
    px,py = pixels[:,0], pixels[:,1]
    in_box = (px>=x1)&(px<=x2)&(py>=y1)&(py<=y2)
    valid_indices = np.where(valid_mask)[0]
    lidar_indices = valid_indices[in_box]
    return pts_xyz[lidar_indices], depth[in_box], pixels[in_box]

sample_id  = '000000'
P2, R0, Tr = parse_calib(f'{CALIB_PATH}/{sample_id}.txt')
pts        = np.fromfile(f'{LIDAR_PATH}/{sample_id}.bin', dtype=np.float32).reshape(-1,4)
pts_xyz    = pts[:,:3]
pixels, depth, valid = project_lidar_to_image(pts_xyz, P2, R0, Tr)

model     = YOLO('yolov8n.pt')
results   = model(f'{IMAGE_PATH}/{sample_id}.png', conf=0.3,
                  classes=[0,1,2,3,5,7], verbose=False)
boxes     = results[0].boxes.xyxy.cpu().numpy()
scores    = results[0].boxes.conf.cpu().numpy()
class_ids = results[0].boxes.cls.cpu().numpy()

# Extract frustum for first detection
box_2d = boxes[0]
person_pts, person_depth, person_pixels = get_lidar_points_in_box(
    box_2d, pixels, depth, pts_xyz, valid)

# Clean background via median filter
median = np.median(person_depth)
mask   = np.abs(person_depth - median) < 2.0
person_pts_clean   = person_pts[mask]
person_depth_clean = person_depth[mask]

print(f'Raw points    : {len(person_pts)}')
print(f'Cleaned points: {len(person_pts_clean)}')
print(f'Distance      : {person_depth_clean.mean():.2f}m')

In [ ]:
# 3D visualization
fig = plt.figure(figsize=(8,6))
ax  = fig.add_subplot(111, projection='3d')
ax.scatter(person_pts_clean[:,0], person_pts_clean[:,1], person_pts_clean[:,2],
           c=person_depth_clean, cmap='jet', s=15)
ax.set_xlabel('X (forward)'); ax.set_ylabel('Y (left)'); ax.set_zlabel('Z (up)')
ax.set_title(f'Frustum LiDAR Points — {person_depth_clean.mean():.1f}m')
plt.tight_layout(); plt.show()